# Feature Engineering for Model Improvement — Titanic

**Raw data is rarely useful as-is.** Features must be created from domain knowledge  
to give the model signal it cannot discover on its own.

**Tasks:**
1. Create `family_size = sibsp + parch + 1`
2. Create `is_alone` (binary feature)
3. Compute survival rate by `is_alone`
4. Explain why engineered features improve ML models

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic').copy()

print(f"Shape: {df.shape}")
print()
# Preview the raw columns we'll use
print("Column meanings:")
print("  sibsp  — number of siblings / spouses aboard")
print("  parch  — number of parents / children aboard")
print()
df[['survived', 'sibsp', 'parch', 'sex', 'pclass']].head(10)

Shape: (891, 15)

Column meanings:
  sibsp  — number of siblings / spouses aboard
  parch  — number of parents / children aboard



,survived,sibsp,parch,sex,pclass
0,0,1,0,male,3
1,1,1,0,female,1
2,1,0,0,female,3
3,1,1,0,female,1
4,0,0,0,male,3
5,0,0,0,male,3
6,0,0,0,male,1
7,0,3,1,male,3
8,1,0,2,female,3
9,1,1,0,female,2


In [9]:
df = sns.load_dataset('titanic').copy()

df['family_size'] = df['sibsp'] + df['parch'] + 1

print(f"Min family size : {df['family_size'].min()}")
print(f"Max family size : {df['family_size'].max()}")
print(f"Mean family size: {df['family_size'].mean():.2f}")
print()


Min family size : 1
Max family size : 11
Mean family size: 1.90



In [11]:
df['is_alone'] = (df['family_size'] == 1).astype(int)   

print("is_alone value counts:")
counts = df['is_alone'].value_counts()
for val, count in counts.items():
    label = 'Travelling alone' if val == 1 else 'With family'
    pct   = count / len(df) * 100
    print(f"  {val} ({label:<20}): {count:>4} passengers  ({pct:.1f}%)")

print()


is_alone value counts:
  1 (Travelling alone    ):  537 passengers  (60.3%)
  0 (With family         ):  354 passengers  (39.7%)



In [19]:
overall_rate = df['survived'].mean()

survival_by_alone = df.groupby('is_alone')['survived'].agg(
    Survival_Rate='mean',
    Survived='sum',
    Total='count'
).rename(index={0: 'With family (0)', 1: 'Alone (1)'})
survival_by_alone['Survival_Rate'] = survival_by_alone['Survival_Rate'].round(4)

print("Survival rate by is_alone:")
print(survival_by_alone.to_string())


print()


Survival rate by is_alone:
                 Survival_Rate  Survived  Total
is_alone                                       
With family (0)         0.5056       179    354
Alone (1)               0.3035       163    537



In [ ]:
# --- Extended engineering: family_group bucket ---
df['family_group'] = pd.cut(
    df['family_size'],
    bins=[0, 1, 4, 20],
    labels=['alone', 'small_family', 'large_family']
)

print("Survival rate by family_group (captures the non-linear curve):")
print()
group_stats = df.groupby('family_group', observed=True)['survived'].agg(
    Survival_Rate='mean',
    Count='count'
)
group_stats['Survival_Rate'] = (group_stats['Survival_Rate'] * 100).round(1).astype(str) + '%'
print(group_stats.to_string())

print()
print("Final engineered features added to the dataset:")
print(df[['sibsp', 'parch', 'family_size', 'is_alone', 'family_group', 'survived']].head(12).to_string())

In [ ]:
# --- Quick model comparison: raw features vs engineered features ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df_model = df[['survived', 'pclass', 'sibsp', 'parch', 'family_size', 'is_alone']].dropna()

y = df_model['survived']

# Raw features
X_raw = df_model[['pclass', 'sibsp', 'parch']]
score_raw = cross_val_score(LogisticRegression(), X_raw, y, cv=5, scoring='accuracy').mean()

# Engineered features
X_eng = df_model[['pclass', 'family_size', 'is_alone']]
score_eng = cross_val_score(LogisticRegression(), X_eng, y, cv=5, scoring='accuracy').mean()

print("Logistic Regression — 5-fold cross-validation accuracy")
print(f"  Raw features    (pclass, sibsp, parch)         : {score_raw*100:.2f}%")
print(f"  Engineered      (pclass, family_size, is_alone): {score_eng*100:.2f}%")
print(f"  Improvement     : +{(score_eng - score_raw)*100:.2f} percentage points")
print()
print("Same number of features (3), but engineered features encode richer signal.")